# PortfolioRiskDiagnosis

This notebook builds an enriched `PortfolioRiskDiagnosis` object from:

- the app-generated diagnosis bundle
- processed structured datasets
- processed unstructured document corpus

The goal is to move the project forward in object fashion, not just dataframe fashion.


## What this enriched version adds

Compared with the first stab, this version now pulls in external data so the diagnosis can carry:

- macro regime context
- fundamental snapshots for the main holding drivers
- narrative evidence from filings and news

This is still diagnosis, not recommendation. It tells us what the current portfolio problem is and what evidence supports that read.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'data_pipeline' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DIAGNOSIS_DIR = ROOT / 'data' / 'processed' / 'diagnosis'
OUTPUT_PATH = DIAGNOSIS_DIR / 'portfolio_risk_diagnosis_enriched.json'

from portfolio_analyzer.diagnosis import portfolio_risk_diagnosis_from_saved_artifacts

print('Diagnosis directory:', DIAGNOSIS_DIR)
print('Exists:', DIAGNOSIS_DIR.exists())


In [ ]:
diagnosis = portfolio_risk_diagnosis_from_saved_artifacts(DIAGNOSIS_DIR)

print('Run ID:', diagnosis.run_id)
print('Observed risk:', diagnosis.observed_risk_score, diagnosis.observed_risk_band)
print('Stated risk:', diagnosis.stated_risk_score, diagnosis.stated_risk_band)
print('Alignment:', diagnosis.alignment)
print('Confidence:', diagnosis.confidence_band)
print()
print('Diagnostic summary:')
print(diagnosis.diagnostic_summary)
print()
print('Macro summary:')
print(diagnosis.macro_context.summary if diagnosis.macro_context else 'No macro context built')


In [ ]:
concerns_df = pd.DataFrame([item.model_dump() for item in diagnosis.top_concerns])
holding_drivers_df = pd.DataFrame([item.model_dump() for item in diagnosis.top_holding_drivers])
sector_drivers_df = pd.DataFrame([item.model_dump() for item in diagnosis.top_sector_drivers])
supporting_metrics_df = pd.DataFrame([item.model_dump() for item in diagnosis.supporting_metrics])
holding_fundamentals_df = pd.DataFrame([item.model_dump() for item in diagnosis.holding_fundamentals])
narrative_evidence_df = pd.DataFrame([item.model_dump() for item in diagnosis.narrative_evidence])
macro_context_df = pd.DataFrame([diagnosis.macro_context.model_dump()]) if diagnosis.macro_context else pd.DataFrame()

display(concerns_df)
display(holding_drivers_df)
display(sector_drivers_df)
display(macro_context_df)
display(holding_fundamentals_df)
display(narrative_evidence_df)
display(supporting_metrics_df[['metric_key', 'group', 'label', 'raw_value', 'score', 'score_readout']])


In [ ]:
diagnosis.model_dump()


## What is still unfinished

This enriched engine is much better grounded, but it is still an early diagnosis layer.

The next improvements should include:

- better metric selection from SEC company facts
- stronger macro-to-risk linkage
- narrative evidence scoring, not just attachment
- concern ranking that uses fundamental and narrative evidence more explicitly
- object relationships into `HoldingRiskContribution` and later `SellCandidate`


In [ ]:
OUTPUT_PATH.write_text(json.dumps(diagnosis.model_dump(), indent=2) + '\n', encoding='utf-8')
print('Saved enriched diagnosis object to:')
print(OUTPUT_PATH)
